

<div style="background: linear-gradient(120deg, #1ca9c9 0%, #40c4d6 60%, #7fe0e0 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Actividad 6:  Datos Faltantes (Imputación Multivariada)</h1>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Laboratorio de Procesamiento de Datos</h3>
    </div>
</div>



Esta actividad consiste en aplicar y comparar dos **métodos de imputación multivariada** en un conjunto de datos numérico. Además de describir cómo funcionan, evaluarás qué tan bien recuperan valores que se ocultaron deliberadamente.

Se utilizará el conjunto **Diabetes** incluido en `scikit-learn` (`load_diabetes`). Contiene **442 observaciones** y **10 carácterísticas numéricas**, sus valores están escalados y las variables presentan relaciones entre sí. No se requiere descargar un archivo. La variable objetivo no se incluye en la imputación.

La primera celda carga las variables completas (`X_completo`), crea una máscara aleatoria reproducible y oculta cerca del 15 % de sus celdas (`X_incompleto`). Conserva `X_completo` y `mascara`: servirán para comparar las imputaciones con los valores reales que se ocultaron. En este ejercicio suponemos que los datos faltantes son **MCAR** (se generan al azar).


In [1]:
from sklearn.datasets import load_diabetes
import numpy as np

# Dataset completo para conservar los valores reales como referencia.
diabetes = load_diabetes(as_frame=True)
X_completo = diabetes.data

# Ocultamos aleatoriamente el 15 % de las celdas; la semilla hace reproducible el ejercicio.
rng = np.random.default_rng(42)
mascara = rng.random(X_completo.shape) < 0.15
X_incompleto = X_completo.mask(mascara)

print(f"Observaciones: {X_completo.shape[0]}")
print(f"Variables predictoras: {X_completo.shape[1]}")
print(f"Valores ocultos: {mascara.sum()} ({mascara.mean():.1%})")
X_incompleto.head()


Observaciones: 442
Variables predictoras: 10
Valores ocultos: 674 (15.2%)


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,NaN,-0.034821,-0.043401,-0.002592,NaN,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,NaN,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,NaN,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,NaN,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,NaN


1. **KNNImputer.** Investiga y explica cómo `sklearn.impute.KNNImputer` estima un valor usando filas vecinas y la distancia euclidiana ajustada a las variables observadas en común. Aplica el método al dataset `X_incompleto`, prueba al menos dos valores de `n_neighbors` (argumento de entrada de KNNImputer) y explica cómo elegiste el mejor.


**¿Cómo funciona `KNNImputer`?**

Para cada fila con valores faltantes, `KNNImputer` busca las `n_neighbors` filas más parecidas (sus *vecinos*) que **sí tengan observado** el valor que falta en esa variable, y lo estima con el promedio de esos vecinos (con `weights='uniform'`, todos pesan igual; con `weights='distance'`, los más cercanos pesan más).

La cercanía se mide con una **distancia euclidiana ajustada para valores faltantes** (`nan_euclidean`). Como las filas tienen huecos en distintas columnas, la distancia solo se calcula con las variables que **ambas filas tienen observadas en común**, y se reescala para compensar las variables que no se pudieron usar:

$$d(x, y) = \sqrt{\; w \sum_{j \in \text{comunes}} (x_j - y_j)^2 \;}, \qquad w = \frac{p}{\#\text{comunes}}$$

donde $p$ es el número total de variables. Así, dos filas que comparten pocas columnas observadas no parecen artificialmente más cercanas solo porque se sumaron menos términos.

In [ ]:
#Código aquí
import pandas as pd
from sklearn.impute import KNNImputer

def rmse_ocultos(X_imputado):
    """RMSE calculado solo sobre las celdas que se ocultaron."""
    real = X_completo.values[mascara]
    estimado = np.asarray(X_imputado)[mascara]
    return np.sqrt(np.mean((real - estimado) ** 2))

# probamos varios valores
resultados_knn = {}
for k in [1, 3, 5, 10, 15, 20]:
    imputador = KNNImputer(n_neighbors=k)
    X_knn = imputador.fit_transform(X_incompleto)
    resultados_knn[k] = rmse_ocultos(X_knn)

tabla_knn = pd.Series(resultados_knn, name="RMSE").rename_axis("n_neighbors")
print(tabla_knn.round(5))

mejor_k = tabla_knn.idxmin()
print(f"\nMejor n_neighbors= {mejor_k} (RMSE = {tabla_knn.min():.5f})")

# imputación final con el mejor k
X_knn = pd.DataFrame(
    KNNImputer(n_neighbors=mejor_k).fit_transform(X_incompleto),
    columns=X_incompleto.columns,
)
print("Valores faltantes restantes=", X_knn.isna().sum().sum())
X_knn.head()

n_neighbors
1     0.05005
3     0.03954
5     0.03706
10    0.03501
15    0.03466
20    0.03495
Name: RMSE, dtype: float64

Mejor n_neighbors= 15 (RMSE = 0.03466)
Valores faltantes restantes= 0


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.016796,-0.034821,-0.043401,-0.002592,0.019364,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.037845,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,0.006904,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.017629,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,0.002512


2. **IterativeImputer.** Investiga y explica cómo `sklearn.impute.IterativeImputer` estima cada variable incompleta a partir de las demás y repite el proceso por rondas. Aplica el método a `X_incompleto`.

**¿Cómo funciona `IterativeImputer`?**

`IterativeImputer` modela cada variable con valores faltantes como función de **todas las demás variables** (estrategia tipo MICE / *chained equations*). El procedimiento es:

1. **Inicialización:** rellena todos los huecos con un valor simple (por defecto, la media de cada columna).
2. **Ronda de imputación:** recorre las columnas con faltantes una por una. Para cada columna $j$, ajusta un regresor (por defecto `BayesianRidge`) usando como variable objetivo la columna $j$ y como predictoras las demás columnas, entrenando solo con las filas donde $j$ sí fue observada. Después predice y reemplaza únicamente las celdas faltantes de $j$.
3. **Repetición:** al terminar de recorrer todas las columnas se completa una *iteración*. Como las predictoras ya tienen mejores estimaciones que la media inicial, se repite el proceso hasta `max_iter` rondas o hasta que los valores imputados dejen de cambiar más que `tol`.

In [4]:
#Código aquí
from sklearn.experimental import enable_iterative_imputer  
from sklearn.impute import IterativeImputer

imp_iter = IterativeImputer(max_iter=10, random_state=42)
X_iter = pd.DataFrame(
    imp_iter.fit_transform(X_incompleto),
    columns=X_incompleto.columns,
)

print("Valores faltantes restantes=", X_iter.isna().sum().sum())
print(f"Iteraciones realizadas= {imp_iter.n_iter_}")
print(f"RMSE IterativeImputer= {rmse_ocultos(X_iter):.5f}")
print(f"RMSE KNNImputer (k={mejor_k}): {rmse_ocultos(X_knn):.5f}")
X_iter.head()

Valores faltantes restantes= 0
Iteraciones realizadas= 10
RMSE IterativeImputer= 0.02962
RMSE KNNImputer (k=15): 0.03466


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.046819,-0.034821,-0.043401,-0.002592,0.003617,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.066116,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.000161,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.043498,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.007514
